# Load RESPOND News Data

This notebook reads all raw country news CSV files from Research Drive via WebDAV. It discovers every file named `*_news.csv` in the project folder, such as `Bulgaria_news.csv`, `Italy_news.csv`, `Netherlands_news.csv`, and `United_Kingdom_news.csv`.

It can also load existing annotation output files from the `annotations/` folder if you want to inspect prior classification results.

In [ ]:
from pathlib import Path
import os

import pandas as pd

from config import RD_BASE_DIR
from dataloader import load_country_news_files_webdav

In [ ]:
# Research Drive location shown in the screenshot:
# ASCOR-FMG-5580-RESPOND-news-data (Projectfolder)/
#
# Configure BASE_URL, USER, and APP_PASSWORD through config_local.py or environment variables.

NEWS_DIR = RD_BASE_DIR
COUNTRIES = None  # None means: discover and load every *_news.csv file.

print(f"Research Drive directory: {NEWS_DIR}")
print("Countries:      all *_news.csv files")

In [ ]:
df = load_country_news_files_webdav(data_dir=NEWS_DIR, countries=COUNTRIES)
print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns.")
df.head()

In [ ]:
df.info()

In [ ]:
# Quick checks that are useful for the raw news data.
for column in ["country", "source.uri", "lang", "dateTime"]:
    if column in df.columns:
        display(df[column].value_counts(dropna=False).rename_axis(column).to_frame("count"))

## Clean And Deduplicate

This section cleans the already-loaded `df` without running the LLM classifier. It creates standardized `article_text`, removes empty/short articles, parses dates, removes duplicates, and saves cleaned outputs locally.

In [ ]:
import hashlib
import re
from pathlib import Path

MIN_WORDS = 80
OUTPUT_ROOT = Path(os.environ.get("RESPOND_OUTPUT_ROOT", Path.home() / "data"))
if OUTPUT_ROOT.exists():
    DEDUP_OUTPUT_DIR = OUTPUT_ROOT / "RESPOND-victims-of-corruption" / "political_corruption_pipeline"
else:
    DEDUP_OUTPUT_DIR = Path("output/political_corruption_pipeline")

TEXT_COLUMN_CANDIDATES = [
    "translated_text",
    "combined_text",
    "body",
    "text",
    "article_text",
    "content",
]

KEEP_CLEANED_COLUMNS = [
    "uri",
    "country",
    "dateTime",
    "date_parsed",
    "year",
    "source.uri",
    "article_text",
    "word_count",
    "text_hash",
    "near_dup_hash",
]

def choose_text_column(dataframe):
    for column in TEXT_COLUMN_CANDIDATES:
        if column in dataframe.columns:
            return column
    raise ValueError("No usable text column found. Expected one of: " + ", ".join(TEXT_COLUMN_CANDIDATES))

def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_for_hash(text):
    text = normalize_text(text).lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def text_hash(text):
    normalized = normalize_for_hash(text)
    return hashlib.md5(normalized.encode("utf-8")).hexdigest()

def cheap_near_duplicate_fingerprint(text, n_tokens=80):
    normalized = normalize_for_hash(text)
    fingerprint = " ".join(normalized.split()[:n_tokens])
    return hashlib.md5(fingerprint.encode("utf-8")).hexdigest()


In [ ]:
df_clean = df.copy()
print(f"Starting rows: {len(df_clean):,}")

text_col = choose_text_column(df_clean)
print(f"Using text column: {text_col}")
df_clean["article_text"] = df_clean[text_col].map(normalize_text)

if "isDuplicate" in df_clean.columns:
    before = len(df_clean)
    is_duplicate = df_clean["isDuplicate"].astype(str).str.lower().isin(["true", "1", "yes"])
    df_clean = df_clean[~is_duplicate].copy()
    print(f"After dropping isDuplicate=True rows: {len(df_clean):,} (-{before - len(df_clean):,})")

before = len(df_clean)
df_clean["word_count"] = df_clean["article_text"].str.split().str.len().fillna(0).astype(int)
df_clean = df_clean[df_clean["article_text"].ne("")].copy()
print(f"After removing missing/empty text: {len(df_clean):,} (-{before - len(df_clean):,})")

before = len(df_clean)
df_clean = df_clean[df_clean["word_count"] >= MIN_WORDS].copy()
print(f"After removing articles with word_count < {MIN_WORDS}: {len(df_clean):,} (-{before - len(df_clean):,})")

if "dateTime" in df_clean.columns:
    date_source = df_clean["dateTime"]
elif "date" in df_clean.columns:
    date_source = df_clean["date"]
else:
    date_source = pd.Series(pd.NaT, index=df_clean.index)

df_clean["date_parsed"] = pd.to_datetime(date_source, errors="coerce", utc=True)
df_clean["year"] = df_clean["date_parsed"].dt.year.astype("Int64")

if "uri" in df_clean.columns:
    before = len(df_clean)
    df_clean = df_clean.drop_duplicates(subset=["uri"], keep="first").copy()
    print(f"After URI dedupe: {len(df_clean):,} (-{before - len(df_clean):,})")

df_clean["text_hash"] = df_clean["article_text"].map(text_hash)
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["text_hash"], keep="first").copy()
print(f"After exact text dedupe: {len(df_clean):,} (-{before - len(df_clean):,})")

df_clean["near_dup_hash"] = df_clean["article_text"].map(cheap_near_duplicate_fingerprint)
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["near_dup_hash"], keep="first").copy()
print(f"After cheap near-duplicate dedupe: {len(df_clean):,} (-{before - len(df_clean):,})")

df_clean.head()


In [ ]:
DEDUP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
keep_columns = [column for column in KEEP_CLEANED_COLUMNS if column in df_clean.columns]

# Avoid writing both a huge combined full-text CSV and huge per-country full-text CSVs.
# The combined file is minimal and useful for denominator checks; the full text
# is saved only in compressed per-country files for later screening/classification.
MINIMAL_COMBINED_COLUMNS = [
    "uri",
    "country",
    "dateTime",
    "date_parsed",
    "year",
    "source.uri",
    "word_count",
    "text_hash",
    "near_dup_hash",
]
minimal_columns = [column for column in MINIMAL_COMBINED_COLUMNS if column in df_clean.columns]

combined_output = DEDUP_OUTPUT_DIR / "all_countries_cleaned_deduped_minimal.csv.gz"
df_clean[minimal_columns].to_csv(combined_output, index=False, compression="gzip")
print(f"Saved minimal combined cleaned file: {combined_output}")

if "country" in df_clean.columns:
    for country, country_df in df_clean.groupby("country", dropna=False):
        safe_country = str(country).replace("/", "_")
        country_output = DEDUP_OUTPUT_DIR / f"{safe_country}_cleaned_deduped.csv.gz"
        country_df[keep_columns].to_csv(country_output, index=False, compression="gzip")
        print(f"Saved {len(country_df):,} rows: {country_output}")


## Optional: Load Annotation Data

Use this when you want the original validation annotations from `config.ANNOTATION_FILE`.

In [ ]:
from dataloader import load_human_annotated_for_translation_webdav

df_annotations = load_human_annotated_for_translation_webdav()
print(f"Loaded {len(df_annotations):,} annotated rows.")
df_annotations.head()